# B3 · Transformer Attention Mechanism

_(placeholder: intro — profile a CPU transformer block, replace the attention bottleneck with three progressively better numba CUDA kernels, measure everything)_

## Setup

_(placeholder: environment — installs, imports, GPU check)_

In [ ]:
# Environment setup -- Colab-ready. torch / numba / matplotlib ship
# preinstalled on Colab GPU runtimes; install anything missing quietly.
import importlib.util
import subprocess
import sys

for mod, pkg in [("numba", "numba"), ("matplotlib", "matplotlib"),
                 ("pynvml", "nvidia-ml-py")]:
    if importlib.util.find_spec(mod) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg],
                       check=True)

import matplotlib.pyplot as plt
import numpy as np
import torch
from numba import cuda

print(f"torch {torch.__version__} | numba CUDA available: {cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("No CUDA GPU -- on Colab: Runtime > Change runtime type > T4 GPU.")
    print("CPU-only cells (correctness gate, Chart 1) still run.")

## Transformer block

_(placeholder: shared skeleton; subclasses swap only the attention step)_

In [ ]:
import torch
import torch.nn as nn
from torch.profiler import record_function


class TransformerBase(nn.Module):
    """One transformer block: embedding -> attention -> FFN.

    Embedding, projections, layer norms and FFN always run in PyTorch on the
    CPU; subclasses only swap out the attention step (the profiled bottleneck).
    """

    def __init__(self, vocab_size=1000, d_model=512, n_heads=4, d_ff=512 * 4):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.n_heads = n_heads
        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, d_model, bias=False)
        self.v_proj = nn.Linear(d_model, d_model, bias=False)

        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Linear(d_ff, d_model)
        )
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

    def attention(self, q, k, v):
        """Scaled dot-product attention: [B, N, D] x3 -> [B, N, D]."""
        raise NotImplementedError()

    def forward(self, x):
        # x: [B, N] token ids
        with record_function("1_embedding"):
            h = self.embedding(x)  # [B, N, D]

        q, k, v = self.q_proj(h), self.k_proj(h), self.v_proj(h)

        with record_function("2_attention_total"):
            attn_out = self.attention(q, k, v)

        h = self.norm1(h + attn_out)

        with record_function("3_ffn"):
            h = self.norm2(h + self.ffn(h))

        return h

## CPU baselines

_(placeholder: python-loop timing baseline + all-PyTorch profiling reference)_

In [ ]:
import math

import torch
from torch.profiler import record_function


def _attend(q, k, v, scale):
    """Attention for one sequence as plain Python lists: [N][D] x3 -> [N][D].

    Both matmuls are the classic naive form — three nested loops, one output
    element per innermost accumulation.
    """
    n, d = len(q), len(q[0])

    with record_function("2a_qk_matmul"):
        scores = [[0.0] * n for _ in range(n)]  # QK^T / sqrt(D)
        for i in range(n):
            for j in range(n):
                s = 0.0
                for x in range(d):
                    s += q[i][x] * k[j][x]
                scores[i][j] = s * scale

    with record_function("2b_softmax"):
        weights = []  # row softmax, max-subtracted for stability
        for row in scores:
            m = max(row)
            exps = [math.exp(s - m) for s in row]
            total = sum(exps)
            weights.append([e / total for e in exps])

    with record_function("2c_value_weighted_sum"):
        out = [[0.0] * d for _ in range(n)]  # weights @ V
        for i in range(n):
            for x in range(d):
                s = 0.0
                for j in range(n):
                    s += weights[i][j] * v[j][x]
                out[i][x] = s
    return out


class CpuPipeline(TransformerBase):
    """CPU pipeline: attention as bare Python loops.

    No BLAS, no SIMD, no threads, no cache blocking — the sequential baseline.
    O(N^2 D) interpreter ops, so the benchmark sweep is capped where its
    runtime stays practical.
    """

    def attention(self, q, k, v):
        scale = q.shape[-1] ** -0.5
        return torch.stack([
            torch.tensor(_attend(q[b].tolist(), k[b].tolist(), v[b].tolist(), scale))
            for b in range(q.shape[0])
        ])


class TorchPipeline(TransformerBase):
    """All-PyTorch pipeline: attention as three torch ops on the CPU.

    The profiling reference for the step-time distribution: every stage runs
    as vectorized ops, so record_function shares reflect *algorithmic* cost,
    not interpreter overhead. Kept unfused (matmul, softmax, matmul) so the
    three attention sub-steps stay separately attributable.
    """

    def attention(self, q, k, v):
        scale = q.shape[-1] ** -0.5
        with record_function("2a_qk_matmul"):
            scores = q @ k.transpose(-2, -1) * scale
        with record_function("2b_softmax"):
            weights = torch.softmax(scores, dim=-1)
        with record_function("2c_value_weighted_sum"):
            return weights @ v

## Profiling & timing helpers

_(placeholder: profiler shares, wall/CUDA-event timing, FLOP/byte floors, footprint, GPU specs)_

In [ ]:
import time

import torch
import torch.nn.functional as F
from numba import cuda
from torch.profiler import ProfilerActivity, profile, record_function

STEPS = ["1_embedding", "2a_qk_matmul", "2b_softmax", "2c_value_weighted_sum", "3_ffn"]
ATTN_STEPS = STEPS[1:4]
# coarse variant: attention as one step, so pipelines without the 2a/2b/2c
# sub-step labels (the GPU versions) are comparable
STEPS_TOTAL = ["1_embedding", "2_attention_total", "3_ffn"]


def _tokens(model, n, batch=1):
    return torch.randint(0, model.embedding.num_embeddings, (batch, n))


def _qkv(model, n):
    with torch.no_grad():
        h = model.embedding(_tokens(model, n))
        return model.q_proj(h), model.k_proj(h), model.v_proj(h)


def sdpa_reference(model, x):
    """Full forward pass with attention done by torch SDPA (correctness oracle)."""
    with torch.no_grad():
        h = model.embedding(x)
        attn = F.scaled_dot_product_attention(model.q_proj(h), model.k_proj(h), model.v_proj(h))
        h = model.norm1(h + attn)
        return model.norm2(h + model.ffn(h))


def step_percentages(model, n, warmup=True, steps=STEPS):
    """% of one forward pass spent in each labelled step (torch.profiler).

    Pass warmup=False for pure-Python models: nothing to warm up, and one
    extra forward costs seconds to minutes.
    """
    x = _tokens(model, n)
    with torch.no_grad():
        if warmup:
            model(x)  # first call pays one-time costs (numba JIT / torch thread pool)
        with profile(activities=[ProfilerActivity.CPU]) as prof:
            with record_function("0_total"):
                model(x)

    def t(key):  # a key can appear more than once -> take max over duplicates
        return max((e.cpu_time_total for e in prof.key_averages() if e.key == key), default=0.0)

    total = t("0_total")
    pct = {s: 100.0 * t(s) / total for s in steps}
    pct["other"] = 100.0 - sum(pct.values())
    return pct


def attention_step_ms(model, n):
    """Absolute ms of each attention sub-step, from one profiled pass over
    the record_function labels inside the model's attention. One pass, no
    warmup — callers warm vectorized models once beforehand. Assumes batch=1 (what _qkv provides):
    key_averages sums duplicate labels, so batch>1 would fold all sequences
    into one number."""
    q, k, v = _qkv(model, n)
    with torch.no_grad():
        with profile(activities=[ProfilerActivity.CPU]) as prof:
            model.attention(q, k, v)

    def t(key):
        return max((e.cpu_time_total for e in prof.key_averages() if e.key == key), default=0.0)

    return {s: t(s) / 1e3 for s in ATTN_STEPS}  # profiler reports microseconds


def attention_ms(model, n, reps=5, warmup=True):
    """Average wall time (ms) of one attention step on precomputed q, k, v.

    GPU pipelines synchronize and copy back to CPU inside `attention`, so
    a wall clock around the call is correct and includes the transfer cost.
    Pass warmup=False for pure-Python models: nothing to JIT, and one extra
    pass costs seconds to minutes.
    """
    q, k, v = _qkv(model, n)
    with torch.no_grad():
        if warmup:
            model.attention(q, k, v)  # warmup / JIT
        t0 = time.perf_counter()
        for _ in range(reps):
            model.attention(q, k, v)
    return (time.perf_counter() - t0) / reps * 1e3


def gpu_specs():
    """Peak fp32 TFLOP/s and memory bandwidth, HW02 roofline style: device
    query for SMs/cc, pynvml for clocks, known-family tables for bus width
    and cores per SM — with a per-family fallback when pynvml is missing."""
    gpu = cuda.get_current_device()
    cc = tuple(gpu.compute_capability)
    sm = gpu.MULTIPROCESSOR_COUNT
    name = gpu.name.decode() if isinstance(gpu.name, bytes) else gpu.name
    cores_per_sm = {(7, 0): 64, (7, 5): 64, (8, 0): 64,
                    (8, 6): 128, (8, 9): 128, (9, 0): 128}.get(cc, 128)
    try:
        import pynvml
        pynvml.nvmlInit()
        h = pynvml.nvmlDeviceGetHandleByIndex(0)
        sm_mhz = pynvml.nvmlDeviceGetMaxClockInfo(h, pynvml.NVML_CLOCK_SM)
        mem_mhz = pynvml.nvmlDeviceGetMaxClockInfo(h, pynvml.NVML_CLOCK_MEM)
        try:
            bus_bits = pynvml.nvmlDeviceGetMemoryBusWidth(h)
        except pynvml.NVMLError:  # per-family fallback
            bus_bits = {(7, 0): 4096, (7, 5): 256, (8, 0): 5120,
                        (8, 6): 384, (8, 9): 384, (9, 0): 6144}.get(cc, 256)
        bw_gbs = mem_mhz * 1e6 * bus_bits / 8 * 2 / 1e9  # DDR: 2 transfers/clock
        pynvml.nvmlShutdown()
    except Exception:  # no pynvml: boost clock + bandwidth from spec sheets
        sm_mhz, bw_gbs = {(7, 5): (1590, 300.0), (8, 0): (1410, 1555.0),
                          (9, 0): (1785, 3900.0)}.get(cc, (1500, 500.0))
    tflops = sm * cores_per_sm * 2 * sm_mhz * 1e6 / 1e12
    return {"name": name, "sm": sm, "tflops": tflops, "bw_gbs": bw_gbs}


def attention_kernel_ms(model, n, reps=5):
    """Device-resident attention time via CUDA events (HW02's cuda_time_ms):
    q/k/v already on the GPU, no transfers in the timed region."""
    q, k, v = (t[0].detach().contiguous().cuda() for t in _qkv(model, n))
    model._attend(q, k, v)  # warmup / JIT
    cuda.synchronize()
    start = cuda.event(timing=True)
    end = cuda.event(timing=True)
    start.record()
    for _ in range(reps):
        model._attend(q, k, v)
    end.record()
    end.synchronize()
    return cuda.event_elapsed_time(start, end) / reps


def attention_kernel_step_ms(model, n, reps=5):
    """Device-resident time (ms) of each attention step via CUDA events.

    Runs the steps in pipeline order so each timed step reads real data
    left behind by the previous one."""
    q, k, v = (t[0].detach().contiguous().cuda() for t in _qkv(model, n))
    N, D = q.shape
    scores = torch.empty((N, N), device=q.device)
    weights = torch.empty((N, N), device=q.device)
    out = torch.empty((N, D), device=q.device)
    steps = {"2a_qk_matmul": lambda: model._step_qkt(q, k, scores),
             "2b_softmax": lambda: model._step_softmax(scores, weights),
             "2c_value_weighted_sum": lambda: model._step_weighted_sum(weights, v, out)}

    times = {}
    for name, fn in steps.items():
        fn()  # warmup / JIT; also fills the buffer the next step reads
        cuda.synchronize()
        start = cuda.event(timing=True)
        end = cuda.event(timing=True)
        start.record()
        for _ in range(reps):
            fn()
        end.record()
        end.synchronize()
        times[name] = cuda.event_elapsed_time(start, end) / reps
    return times


def attention_flops(model, n):
    """FLOP count of one [N, D] attention, from torch's own flop_counter
    (torch.utils.flop_counter) — the per-matmul formula is torch's, not a
    hand-rolled 4*N^2*D. Counts the two matmuls the kernels actually do:
    scores = Q @ K^T and out = weights @ V, each 2*N^2*D FLOPs. The softmax's
    elementwise ops are not FLOP-counted (standard roofline convention). The
    math is identical for V1/V2/V3, so one count serves every version.

    Runs on CPU — counting is analytic, it just needs the ops to dispatch."""
    from torch.utils.flop_counter import FlopCounterMode
    d = model.embedding.embedding_dim
    q = torch.zeros(1, n, d)
    with torch.no_grad(), FlopCounterMode(display=False) as fc:
        w = torch.bmm(q, q.transpose(-1, -2))  # Q @ K^T  -> [1, N, N]
        torch.bmm(w.softmax(-1), q)            # weights @ V -> [1, N, D]
    return fc.get_total_flops()


def attention_bytes(model, n):
    """Compulsory DRAM traffic (bytes) of one [N, D] attention: read Q, K, V
    and write the output, each element touched once. Byte width comes from the
    tensor (element_size), not a magic constant. This is the roofline's
    lower-bound traffic; real kernels move more — V1 re-reads its operands per
    output element, V2/V3 far less.
    ponytail: compulsory-traffic model, so V1/V2/V3 share one x-position and
    differ only in how close to the ceiling they land. Measured DRAM bytes —
    which expose the reuse gap directly — come from src/dram_profile.py
    (Nsight Compute dram__bytes.sum; root-gated counters)."""
    probe = torch.empty(n, model.embedding.embedding_dim)
    return 4 * probe.numel() * probe.element_size()  # Q, K, V in + out out


def _peak_extra_bytes(call):
    """Torch-allocator high-water mark (bytes) over `call`, beyond what is
    already allocated. None on OOM — the honest datapoint at the top of the
    footprint sweep."""
    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()
    base = torch.cuda.memory_allocated()
    try:
        with torch.no_grad():
            call()
        torch.cuda.synchronize()
        return torch.cuda.max_memory_allocated() - base
    except torch.OutOfMemoryError:
        torch.cuda.empty_cache()
        return None


def attention_peak_extra_bytes(model, n):
    """Peak extra GPU memory of one attention pass beyond its q/k/v inputs:
    the N x N intermediates for the unfused versions, ~the [N, D] output for
    the fused ones."""
    q, k, v = (t[0].detach().contiguous().cuda() for t in _qkv(model, n))
    return _peak_extra_bytes(lambda: model._attend(q, k, v))


def bench_attention(classes, seq_lens, reps=5):
    """Attention-step time for each pipeline class at each sequence length."""
    times = {name: [] for name in classes}
    for n in seq_lens:
        for name, cls in classes.items():
            model = cls().eval()
            times[name].append(attention_ms(model, n, reps))
            if torch.cuda.is_available():
                torch.cuda.empty_cache()  # drop cached N x N blocks between runs
    return times

## CPU correctness gate

_(placeholder: both CPU models vs the torch SDPA oracle)_

In [ ]:
# Correctness gate: both CPU models must match torch's SDPA oracle before
# they can serve as profiling / timing references.
torch.manual_seed(0)
cpu_model = CpuPipeline().eval()
x = torch.randint(0, 1000, (2, 16))
with torch.no_grad():
    for m in (cpu_model, TorchPipeline().eval()):
        # CpuPipeline: fp64 python floats vs fp32; TorchPipeline: unfused vs fused
        torch.testing.assert_close(m(x), sdpa_reference(m, x),
                                   atol=1e-4, rtol=1e-3)
print("CpuPipeline & TorchPipeline match the torch SDPA oracle")

## Chart 1 — CPU bottleneck profile

_(placeholder: share of forward-pass time per step vs N)_

In [ ]:
# Chart 1 -- share of forward-pass time per step (all-PyTorch pipeline)
import matplotlib.pyplot as plt
import numpy as np

PROFILE_LENS = [2 ** x for x in range(6, 15)]
torch_model = TorchPipeline().eval()
pcts = [step_percentages(torch_model, n) for n in PROFILE_LENS]

parts = STEPS + ["other"]
names = {"1_embedding": "embedding", "2a_qk_matmul": "attn: QK$^T$",
         "2b_softmax": "attn: softmax", "2c_value_weighted_sum": "attn: weighted sum",
         "3_ffn": "FFN", "other": "other (projections, norms)"}
colors = {"1_embedding": "tab:blue", "2a_qk_matmul": "#ffc04d", "2b_softmax": "tab:orange",
          "2c_value_weighted_sum": "#b35a00", "3_ffn": "tab:green", "other": "lightgray"}

fig, ax = plt.subplots(figsize=(9, 5))
xpos = np.arange(len(PROFILE_LENS))
bottom = np.zeros(len(PROFILE_LENS))
for part in parts:
    vals = np.array([p[part] for p in pcts])
    ax.bar(xpos, vals, 0.6, bottom=bottom, label=names[part], color=colors[part])
    bottom += vals

for xi, p in zip(xpos, pcts):  # annotate total attention share
    attn = sum(p[s] for s in ATTN_STEPS)
    ax.text(xi, 101, f"attn {attn:.0f}%", ha="center", fontsize=8)

ax.set_xticks(xpos, PROFILE_LENS, rotation=45)
ax.set_xlabel("sequence length N")
ax.set_ylabel("% of forward-pass time")
ax.set_ylim(0, 108)
ax.set_title("All-PyTorch pipeline: where a forward pass spends its time")
ax.legend(loc="center left", bbox_to_anchor=(1.01, 0.5))
plt.tight_layout()
plt.show()

## GPU pipeline base

_(placeholder: CPU<->GPU transfer handling around the attention kernels)_

In [ ]:
import torch
from numba import cuda


class GpuPipeline(TransformerBase):
    """Base for the GPU versions: handles CPU<->GPU transfer around `_attend`.

    The measured attention time deliberately includes the transfer cost —
    that is the true price of swapping the kernel into the CPU pipeline.
    """

    def attention(self, q, k, v):
        # detach: torch refuses __cuda_array_interface__ on tensors that
        # require grad, and this pipeline is inference-only anyway
        out = torch.empty_like(q)
        for b in range(q.shape[0]):
            res = self._attend(q[b].detach().contiguous().cuda(),
                               k[b].detach().contiguous().cuda(),
                               v[b].detach().contiguous().cuda())
            cuda.synchronize()
            out[b] = res.cpu()
        return out

    def _attend(self, q, k, v):
        """Attention for one sequence, on device: [N, D] x3 -> [N, D].

        Default: the unfused three-step pipeline, each step a separately
        timeable method. Fused versions override _attend wholesale.
        """
        N, D = q.shape
        scores = torch.empty((N, N), device=q.device)
        weights = torch.empty((N, N), device=q.device)
        out = torch.empty((N, D), device=q.device)
        self._step_qkt(q, k, scores)
        self._step_softmax(scores, weights)
        self._step_weighted_sum(weights, v, out)
        return out

    def _step_qkt(self, q, k, scores):
        """scores = (q @ k^T) / sqrt(D), on device."""
        raise NotImplementedError()

    def _step_softmax(self, scores, weights):
        """weights = row softmax of scores, on device."""
        raise NotImplementedError()

    def _step_weighted_sum(self, weights, v, out):
        """out = weights @ v, on device."""
        raise NotImplementedError()

## GPU V1 — naive three-kernel

_(placeholder: one thread per output element, everything through global memory)_

In [ ]:
import math

import numpy as np
from numba import cuda, float32


@cuda.jit
def _matmul(a, b, scale, out):
    """out = scale * (a @ b) — one thread per output element, all reads from
    global memory. The scale rides the epilogue for free: a separate scaling
    launch would re-read and re-write the whole output matrix.

    threadIdx.x walks the output column (the contiguous axis) so warp reads
    of b are coalesced; a[i, kk] is a broadcast within the warp.
    """
    j, i = cuda.grid(2)
    if i < out.shape[0] and j < out.shape[1]:
        acc = float32(0.)
        for kk in range(a.shape[1]):
            acc += a[i, kk] * b[kk, j]
        out[i, j] = acc * scale


TPB_V1 = 256  # threads cooperating on one row


@cuda.jit
def _softmax(x, out):
    """Row softmax, one block per row: max pass, sum pass, write pass.

    Each thread strides over its slice of the row; the block reduces the
    per-thread max and sum in shared memory. Still three reads of the row
    (naive), but the row is now processed cooperatively instead of by a
    single thread.
    """
    row = cuda.blockIdx.x
    tid = cuda.threadIdx.x
    n = x.shape[1]

    sm = cuda.shared.array(TPB_V1, float32)
    sd = cuda.shared.array(TPB_V1, float32)

    # row max for numerical stability (avoid exp overflow)
    m = x[row, tid] if tid < n else float32(-3.0e38)
    for j in range(tid + TPB_V1, n, TPB_V1):
        if x[row, j] > m:
            m = x[row, j]
    sm[tid] = m
    cuda.syncthreads()

    stride = TPB_V1 // 2
    while stride > 0:
        if tid < stride and sm[tid + stride] > sm[tid]:
            sm[tid] = sm[tid + stride]
        cuda.syncthreads()
        stride //= 2
    m = sm[0]

    d = float32(0.)
    for j in range(tid, n, TPB_V1):
        d += math.exp(x[row, j] - m)
    sd[tid] = d
    cuda.syncthreads()

    stride = TPB_V1 // 2
    while stride > 0:
        if tid < stride:
            sd[tid] += sd[tid + stride]
        cuda.syncthreads()
        stride //= 2
    denom = sd[0]

    for j in range(tid, n, TPB_V1):
        out[row, j] = math.exp(x[row, j] - m) / denom


def _grid2d(rows, cols, tpb=(16, 16)):
    # grid x covers columns, grid y covers rows (matches j, i = cuda.grid(2))
    return (math.ceil(cols / tpb[0]), math.ceil(rows / tpb[1])), tpb


class GpuV1(GpuPipeline):
    """V1 — naive three-launch attention: scaled QK^T matmul, softmax,
    weighted sum. One thread per output element, everything through
    global memory.

    Every intermediate, including the full N x N score matrix, round-trips
    through global memory between kernels, and the softmax reads each row
    three times.
    """

    def _step_qkt(self, q, k, scores):
        N, D = q.shape
        bpg, tpb = _grid2d(N, N)
        # np.float32 keeps the kernel arithmetic in fp32 (a python float is typed float64)
        _matmul[bpg, tpb](q, k.t().contiguous(), np.float32(D ** -0.5), scores)

    def _step_softmax(self, scores, weights):
        _softmax[scores.shape[0], TPB_V1](scores, weights)

    def _step_weighted_sum(self, weights, v, out):
        bpg, tpb = _grid2d(*out.shape)
        _matmul[bpg, tpb](weights, v, np.float32(1.0), out)

## GPU V2 — tiled QK$^T$ + online softmax

_(placeholder: shared-memory tiles + single-pass softmax)_

In [ ]:
import math

import numpy as np
from numba import cuda, float32


TILE = 16          # tile edge for the shared-memory matmuls
TILE_P = TILE + 1  # padded tile width: numba wants a plain constant in
                   # cuda.shared.array, and the pad avoids bank conflicts
TPB_V2 = 128   # threads cooperating on one row in the online softmax

# finite stand-in for -inf: avoids (-inf) - (-inf) = nan when merging two
# partials that saw no elements
_NEG_BIG = float32(-3.0e38)


@cuda.jit
def _qkt_tiled(q, k, scale, out):
    """out[i, j] = scale * dot(q[i], k[j]) — QK^T with both operand tiles
    staged in shared memory; the 1/sqrt(D) scaling is fused into the epilogue.

    threadIdx.x walks the contiguous axis of q and k, so all global loads are
    coalesced; the +1 column pad keeps the sk reads free of bank conflicts.
    """
    sq = cuda.shared.array((TILE, TILE_P), float32)
    sk = cuda.shared.array((TILE, TILE_P), float32)

    tx = cuda.threadIdx.x
    ty = cuda.threadIdx.y
    j = cuda.blockIdx.x * TILE + tx   # output column
    i = cuda.blockIdx.y * TILE + ty   # output row
    jr = cuda.blockIdx.x * TILE + ty  # k row staged by this thread
    N, D = q.shape

    acc = float32(0.)
    for t in range(0, D, TILE):
        sq[ty, tx] = q[i, t + tx] if i < N and t + tx < D else float32(0.)
        sk[ty, tx] = k[jr, t + tx] if jr < N and t + tx < D else float32(0.)
        cuda.syncthreads()
        for kk in range(TILE):
            acc += sq[ty, kk] * sk[tx, kk]
        cuda.syncthreads()

    if i < N and j < N:
        out[i, j] = acc * scale


@cuda.jit
def _matmul_tiled(a, b, out):
    """out = a @ b with shared-memory tiling (used for weights @ V)."""
    sa = cuda.shared.array((TILE, TILE_P), float32)
    sb = cuda.shared.array((TILE, TILE_P), float32)

    tx = cuda.threadIdx.x
    ty = cuda.threadIdx.y
    j = cuda.blockIdx.x * TILE + tx   # output column
    i = cuda.blockIdx.y * TILE + ty   # output row
    M, K = a.shape
    P = b.shape[1]

    acc = float32(0.)
    for t in range(0, K, TILE):
        sa[ty, tx] = a[i, t + tx] if i < M and t + tx < K else float32(0.)
        sb[ty, tx] = b[t + ty, j] if t + ty < K and j < P else float32(0.)
        cuda.syncthreads()
        for kk in range(TILE):
            acc += sa[ty, kk] * sb[kk, tx]
        cuda.syncthreads()

    if i < M and j < P:
        out[i, j] = acc


@cuda.jit
def _softmax_online(x, out):
    """Row softmax in two reads of the row instead of three.

    One block per row. Each thread keeps a running (max, sum) over its slice,
    rescaling the sum whenever the max moves (log-sum-exp trick), so max and
    sum come out of a single pass. Partials merge in shared memory, then a
    second pass normalises and writes. All accesses are coalesced (adjacent
    threads read adjacent columns).
    """
    row = cuda.blockIdx.x
    tid = cuda.threadIdx.x
    n = x.shape[1]

    sm = cuda.shared.array(TPB_V2, float32)
    sl = cuda.shared.array(TPB_V2, float32)

    m = _NEG_BIG
    l = float32(0.)
    for j in range(tid, n, TPB_V2):
        val = x[row, j]
        if val > m:
            l *= math.exp(m - val)  # rescale the sum accumulated so far
            m = val
        l += math.exp(val - m)
    sm[tid] = m
    sl[tid] = l
    cuda.syncthreads()

    stride = TPB_V2 // 2
    while stride > 0:
        if tid < stride:
            m2 = sm[tid + stride]
            l2 = sl[tid + stride]
            if m2 > sm[tid]:
                sl[tid] = sl[tid] * math.exp(sm[tid] - m2) + l2
                sm[tid] = m2
            else:
                sl[tid] += l2 * math.exp(m2 - sm[tid])
        cuda.syncthreads()
        stride //= 2

    m = sm[0]
    l = sl[0]
    for j in range(tid, n, TPB_V2):
        out[row, j] = math.exp(x[row, j] - m) / l


class GpuV2(GpuPipeline):
    """V2 — tiled QK^T in shared memory + online softmax.

    Still materialises the N x N score matrix, but each element of Q and K is
    read from global memory TILE (16x) times less, and the softmax saves one
    full read of the score matrix.
    """

    def _step_qkt(self, q, k, scores):
        N, D = q.shape
        bpg = (math.ceil(N / TILE), math.ceil(N / TILE))
        # np.float32 keeps the kernel arithmetic in fp32 (a python float is typed float64)
        _qkt_tiled[bpg, (TILE, TILE)](q, k, np.float32(1.0 / D ** 0.5), scores)

    def _step_softmax(self, scores, weights):
        _softmax_online[scores.shape[0], TPB_V2](scores, weights)

    def _step_weighted_sum(self, weights, v, out):
        N, D = out.shape
        bpg = (math.ceil(D / TILE), math.ceil(N / TILE))  # (cols, rows)
        _matmul_tiled[bpg, (TILE, TILE)](weights, v, out)

## GPU V3 — fused FlashAttention-1

_(placeholder: one kernel, resident Q tile, online softmax; the N×N matrix never exists)_

In [ ]:
import math
from functools import lru_cache

import numpy as np
import torch
from numba import cuda, float32


TILE = 16      # tile edge: 16 query rows x 16 key columns per block
TILE_P = 17    # padded row width for the score tile (dodges bank conflicts)

# finite stand-in for -inf on masked keys: exp() of it is 0 and it dodges the
# (-inf) - (-inf) = nan trap when a tile holds no real keys
_NEG_BIG = float32(-3.0e38)
_M_INIT = float32(-math.inf)


@lru_cache(maxsize=None)
def _flash_kernel(D):
    """Build the FlashAttention-1 forward kernel for a fixed model width D —
    numba wants compile-time-constant shared/local array shapes."""
    assert D % TILE == 0, "flash kernel assumes d_model is a multiple of 16"
    CH = D // TILE  # output columns owned by each thread (cols tx, tx+16, ...)
    D_P = D + 1     # padded Q row: plain name, same shared.array constraint

    @cuda.jit
    def flash(q, k, v, scale, out):
        tx = cuda.threadIdx.x          # key column within the tile
        ty = cuda.threadIdx.y          # query row within the tile
        base = cuda.blockIdx.x * TILE  # first query row of this block
        N = q.shape[0]

        qs = cuda.shared.array((TILE, D_P), float32)    # resident Q tile
        ps = cuda.shared.array((TILE, TILE_P), float32)  # S, then P~ = exp(S - m~)
        ks = cuda.shared.array((TILE, TILE_P), float32)  # K d-chunk (V2's idiom)
        vs = cuda.shared.array((TILE, TILE_P), float32)  # V column-chunk

        # Stage the block's Q tile once (each thread strides one row); it
        # stays on-chip for the whole K/V sweep — the FlashAttention move
        # that makes Q traffic O(N D) instead of O(N^2 D).
        for c in range(tx, D, TILE):
            qs[ty, c] = q[base + ty, c] if base + ty < N else float32(0.)
        cuda.syncthreads()

        acc = cuda.local.array(CH, float32)  # O[base+ty, tx::TILE] — kept
        for c in range(CH):                  # normalised after every tile (FA-1)
            acc[c] = float32(0.)
        m = _M_INIT      # running row max
        l = float32(0.)  # running row sum of exponentials

        for jt in range(0, N, TILE):
            # S[ty, tx] = scale * dot(q[base+ty], k[jt+tx]) — K staged in
            # 16-wide d-chunks in shared memory, exactly V2's tiling idiom
            s = float32(0.)
            for dt in range(0, D, TILE):
                ks[ty, tx] = k[jt + ty, dt + tx] if jt + ty < N else float32(0.)
                cuda.syncthreads()
                for j in range(TILE):
                    s += qs[ty, dt + j] * ks[tx, j]
                cuda.syncthreads()
            s *= scale
            if jt + tx >= N:
                s = _NEG_BIG  # masked key: exp() contributes exactly 0
            ps[ty, tx] = s
            cuda.syncthreads()

            # tile row max: every lane scans its row's 16 scores — redundant
            # but uniform, so no divergence and no reduction machinery
            mt = _NEG_BIG
            for j in range(TILE):
                if ps[ty, j] > mt:
                    mt = ps[ty, j]
            p = math.exp(s - mt)  # P~ uses the *tile* max (FA-1's Algorithm 1)
            cuda.syncthreads()
            ps[ty, tx] = p
            cuda.syncthreads()
            lt = float32(0.)      # tile row sum, same redundant scan
            for j in range(TILE):
                lt += ps[ty, j]

            # FA-1 online-softmax merge: fold the tile's (m~, l~) into the
            # running (m, l) and rescale AND renormalise the output block this
            # very tile — no deferral; acc always holds the softmax of
            # everything seen so far. First tile: exp(-inf - m_new) = 0.
            mn = mt if mt > m else m
            alpha = math.exp(m - mn)
            beta = math.exp(mt - mn)
            ln = l * alpha + lt * beta
            scale_old = l * alpha / ln  # rescales the already-normalised acc
            scale_new = beta / ln       # normalises this tile's P~ V

            # acc = scale_old * acc + scale_new * (P~ @ V_tile); V staged one
            # 16-column chunk at a time, same idiom
            for c in range(CH):
                vs[ty, tx] = v[jt + ty, c * TILE + tx] if jt + ty < N else float32(0.)
                cuda.syncthreads()
                pv = float32(0.)
                for j in range(TILE):
                    pv += ps[ty, j] * vs[j, tx]
                acc[c] = acc[c] * scale_old + pv * scale_new
                cuda.syncthreads()
            m = mn
            l = ln

        # acc is already the normalised softmax-weighted sum — write as-is
        if base + ty < N:
            for c in range(CH):
                out[base + ty, c * TILE + tx] = acc[c]

    return flash


class GpuV3(GpuPipeline):
    """V3 — FlashAttention-1 forward: tiled + online softmax + one fused
    kernel, and the N x N score matrix never exists in any memory.

    One thread block per 16 query rows: the Q tile is staged in shared memory
    once and stays resident while the block sweeps the K/V tiles; K and V are
    staged 16-wide chunk by chunk with the same shared-memory idiom V2 uses —
    no prefetching, no unrolling, no micro-tiles. Per 16x16 tile the block
    computes the score tile, takes its row max and sum, and applies
    Algorithm 1's update: the running (max, sum) statistics merge with the
    tile's and the output block is rescaled and renormalised immediately, so
    it always holds the exact softmax-weighted sum of every key seen so far.
    Extra footprint is one [N, D] output — O(N) — and Q/K/V are each read
    once per Q-block sweep.
    """

    def _attend(self, q, k, v):
        N, D = q.shape
        out = torch.empty((N, D), device=q.device)
        kernel = _flash_kernel(D)
        # np.float32 keeps the kernel arithmetic in fp32: a python float
        # scale is typed float64 and would silently promote the hot loops
        kernel[math.ceil(N / TILE), (TILE, TILE)](
            q, k, v, np.float32(1.0 / D ** 0.5), out)
        return out

## GPU correctness gate

_(placeholder: each GPU version end-to-end vs torch SDPA)_

In [ ]:
# End-to-end correctness of each GPU version against torch SDPA
if cuda.is_available():
    x = torch.randint(0, 1000, (1, 128))
    for cls in (GpuV1, GpuV2, GpuV3):
        model = cls().eval()
        with torch.no_grad():
            torch.testing.assert_close(model(x), sdpa_reference(model, x),
                                       atol=1e-4, rtol=1e-3)
        print(f"{cls.__name__}: matches torch SDPA within 1e-4")
else:
    print("No CUDA device -- skipping GPU correctness checks")

## Chart 1b — step-time distribution shift

_(placeholder: same buckets across all four pipelines; the bottleneck moves back)_

In [ ]:
# Chart 1b -- step-time distribution per pipeline (coarse buckets, all comparable)
if cuda.is_available():
    # torch panel reuses Chart 1's profile: the sub-steps sum to the attention total
    dist = {"all-PyTorch (CPU)": [
        {"1_embedding": p["1_embedding"],
         "2_attention_total": sum(p[s] for s in ATTN_STEPS),
         "3_ffn": p["3_ffn"], "other": p["other"]} for p in pcts]}
    for title, m in (("V1 naive (GPU attention)", GpuV1().eval()),
                     ("V2 tiled (GPU attention)", GpuV2().eval()),
                     ("V3 flash (GPU attention)", GpuV3().eval())):
        dist[title] = [step_percentages(m, n, steps=STEPS_TOTAL)
                       for n in PROFILE_LENS]
    torch.cuda.empty_cache()  # drop the dead N x N blocks the GPU passes leave cached

    parts = STEPS_TOTAL + ["other"]
    names = {"1_embedding": "embedding", "2_attention_total": "attention",
             "3_ffn": "FFN", "other": "other (projections, norms)"}
    colors = {"1_embedding": "tab:blue", "2_attention_total": "tab:orange",
              "3_ffn": "tab:green", "other": "lightgray"}

    fig, axes = plt.subplots(1, 4, figsize=(19, 5), sharey=True)
    xpos = np.arange(len(PROFILE_LENS))
    for ax, (title, ps) in zip(axes, dist.items()):
        bottom = np.zeros(len(PROFILE_LENS))
        for part in parts:
            vals = np.array([p[part] for p in ps])
            ax.bar(xpos, vals, 0.6, bottom=bottom, label=names[part], color=colors[part])
            bottom += vals
        for xi, p in zip(xpos, ps):  # annotate the attention share
            ax.text(xi, 101, f"{p['2_attention_total']:.0f}%", ha="center", fontsize=8)
        ax.set_xticks(xpos, PROFILE_LENS, rotation=45)
        ax.set_xlabel("sequence length N")
        ax.set_title(title)
        ax.set_ylim(0, 108)
    axes[0].set_ylabel("% of forward-pass time")
    axes[-1].legend(loc="center left", bbox_to_anchor=(1.02, 0.5))
    plt.suptitle("Where the forward pass goes once attention moves to the GPU (attention % annotated)")
    plt.tight_layout()
    plt.show()
else:
    print("No CUDA device -- skipping the combined step-share chart")

## Benchmark — attention step time

_(placeholder: wall time incl. CPU<->GPU transfers; CPU feasibility guard)_

In [ ]:
SEQ_LENS = [2 ** x for x in range(6, 13)]   # speedup sweep: capped by the CPU
KERNEL_LENS = [2 ** x for x in range(6, 16)]   # GPU-only analyses sweep to 32768,
                                               # the last N before V1/V2 OOM

if cuda.is_available():
    classes = {"V1 naive": GpuV1,
               "V2 tiled + online softmax": GpuV2,
               "V3 flash fused": GpuV3,
               }
    times = bench_attention(classes, SEQ_LENS, reps=3)
    torch.cuda.empty_cache()  # drop cached blocks so the sweep leaves no footprint

    # feasibility guard: the baseline is interpreted python with O(N^2 D)
    # cost -- calibrate one pass, project the sweep top against the cell
    # timeout and available RAM
    import os
    cal_ms = attention_ms(cpu_model, 256, reps=1, warmup=False)
    proj_s = cal_ms * (SEQ_LENS[-1] / 256) ** 2 / 1e3
    ram = os.sysconf("SC_PHYS_PAGES") * os.sysconf("SC_PAGE_SIZE")
    need = 2 * SEQ_LENS[-1] ** 2 * 32  # scores + weights as python float lists
    if proj_s > 3300 or need > 0.6 * ram:
        print(f"WARNING: N = {SEQ_LENS[-1]} may not be feasible on the python "
              f"baseline: projected ~{proj_s / 60:.0f} min per attention pass, "
              f"~{need / 1e9:.1f} GB of {ram / 1e9:.0f} GB RAM")

    # CPU measured last (slowest by far): one profiled pass per point yields
    # the per-step times (Chart 2's baselines) and their sum, the CPU column
    cpu_steps = [attention_step_ms(cpu_model, n) for n in SEQ_LENS]
    times = {"CPU": [sum(d.values()) for d in cpu_steps], **times}

    header = f"{'N':>6} " + "".join(f"{name:>28}" for name in times)
    print(header + "\n" + "-" * len(header))
    for i, n in enumerate(SEQ_LENS):
        print(f"{n:>6} " + "".join(f"{times[name][i]:>25.2f} ms" for name in times))
else:
    print("No CUDA device -- skipping the GPU benchmark")

COLORS = {"CPU": "tab:red",
          "V1 naive": "tab:blue", "V2 tiled + online softmax": "tab:orange",
          "V3 flash fused": "tab:green"}

## Kernel-only timing

_(placeholder: device-resident CUDA-event sweep to N = 32768)_

In [ ]:
# Kernel-only timing sweep (CUDA events, device-resident) -- GPU-only, so it
# runs past the CPU-capped benchmark to N = 2^15 for Charts 3-5
if cuda.is_available():
    specs = gpu_specs()
    print(f"GPU: {specs['name']} | {specs['sm']} SMs")

    kmodels = {"V1 naive": GpuV1().eval(),
               "V2 tiled + online softmax": GpuV2().eval(),
               "V3 flash fused": GpuV3().eval(),
               }
    ktimes = {name: [attention_kernel_ms(m, n, reps=10) for n in KERNEL_LENS]
              for name, m in kmodels.items()}
    # per-step kernel times (QK^T, softmax, weighted sum) for Chart 2 --
    # only the unfused versions expose the three step methods (V3 is one kernel)
    ksteps = {name: [attention_kernel_step_ms(m, n, reps=10) for n in KERNEL_LENS]
              for name, m in kmodels.items() if "_step_qkt" in type(m).__dict__}
    torch.cuda.empty_cache()  # drop the dead N x N blocks V1/V2 leave cached

    header = f"{'N':>6}" + "".join(f"{name:>30}" for name in ktimes)
    print("\n" + header + "   (kernel-only ms)\n" + "-" * len(header))
    for i, n in enumerate(KERNEL_LENS):
        print(f"{n:>6}" + "".join(f"{ktimes[name][i]:>27.3f} ms" for name in ktimes))
else:
    print("No CUDA device -- skipping the kernel metrics")

## Chart 2 — speedup over the CPU baseline

_(placeholder: per sub-step and whole-step speedups)_

In [ ]:
# Chart 2 -- speedup over the python-loop CPU attention: per sub-step for
# the unfused versions, then the whole step for all three (V3 is a single
# fused kernel, so whole-step granularity only)
if cuda.is_available():
    step_names = {"2a_qk_matmul": "QK$^T$ (+scale)", "2b_softmax": "softmax",
                  "2c_value_weighted_sum": "weighted sum"}
    fig, axes = plt.subplots(1, 4, figsize=(19, 5), sharey=True)
    xpos = np.arange(len(SEQ_LENS))
    w = 0.76 / len(ksteps)
    offs = (np.arange(len(ksteps)) - (len(ksteps) - 1) / 2) * w
    for ax, (step, title) in zip(axes, step_names.items()):
        for off, (name, ks) in zip(offs, ksteps.items()):
            kmap = dict(zip(KERNEL_LENS, ks))
            speedups = [c[step] / kmap[n][step] for c, n in zip(cpu_steps, SEQ_LENS)]
            ax.bar(xpos + off, speedups, w, color=COLORS[name], label=name)
        ax.set_yscale("log")
        ax.set_xticks(xpos, SEQ_LENS, rotation=45)
        ax.set_xlabel("sequence length N")
        ax.set_title(title)
        ax.grid(True, axis="y", which="both", alpha=0.3)

    ax = axes[-1]
    cpu_total = [sum(c.values()) for c in cpu_steps]
    w4 = 0.8 / len(ktimes)
    offs4 = (np.arange(len(ktimes)) - (len(ktimes) - 1) / 2) * w4
    for off, (name, kt) in zip(offs4, ktimes.items()):
        kmap = dict(zip(KERNEL_LENS, kt))
        speedups = [c / kmap[n] for c, n in zip(cpu_total, SEQ_LENS)]
        ax.bar(xpos + off, speedups, w4, color=COLORS[name], label=name)
    ax.set_yscale("log")
    ax.set_xticks(xpos, SEQ_LENS, rotation=45)
    ax.set_xlabel("sequence length N")
    ax.set_title("whole attention step (all versions)")
    ax.grid(True, axis="y", which="both", alpha=0.3)
    ax.legend(fontsize=8)

    axes[0].set_ylabel("speedup over CPU (x)")
    axes[0].legend()
    plt.suptitle("Speedup over the python-loop CPU attention (kernel-only time)")
    plt.tight_layout()
    plt.show()
else:
    print("No CUDA device -- skipping the speedup chart")

## Chart 2b — GPU versions head-to-head

_(placeholder: kernel-only speedup over V1 across the full sweep)_

In [ ]:
# Chart 2b -- the three GPU rungs head-to-head (kernel-only), swept past the
# CPU cap to N = 32768 as speedup over V1 -- the crossovers Chart 2's
# CPU-relative bars can't resolve
if cuda.is_available():
    fig, ax = plt.subplots(figsize=(9, 5))
    for name, kt in ktimes.items():
        ax.plot(KERNEL_LENS, [v1 / t for v1, t in zip(ktimes["V1 naive"], kt)],
                marker="o", color=COLORS[name], label=name)
    ax.set_xscale("log", base=2)
    ax.set_xticks(KERNEL_LENS, KERNEL_LENS, rotation=45)
    ax.set_xlabel("sequence length N")
    ax.set_ylabel("speedup over V1 naive (x)")
    ax.grid(True, which="both", alpha=0.3)
    ax.legend(fontsize=8)
    ax.set_title("GPU versions head-to-head (speedup over V1, kernel-only time)")
    plt.tight_layout()
    plt.show()
else:
    print("No CUDA device -- skipping the head-to-head chart")

## Measured DRAM counters (embedded)

_(placeholder: Nsight Compute bytes/FLOPs per pass, T4, collected offline)_

In [ ]:
# Measured Nsight Compute counters, one attention pass per (version, N),
# on a Tesla T4 (Colab's default GPU). dram_bytes = dram__bytes.sum, every
# transaction crossing the DRAM<->L2 boundary, refetches included; flops =
# fadd + fmul + 2*ffma SASS thread instructions; kernel_ms = ncu's
# base-clock duration (reference only -- the figures pair the counts with
# the boost-clock CUDA-event sweep above). Embedded inline because the
# counters are root-gated (one-off `sudo ncu` collection; the collection
# script is src/dram_profile.py in the repo).
DRAM_PROFILE = {
 "v1": [
  {"n": 64, "dram_bytes": 995840, "kernel_ms": 0.21235199999999999, "flops": 8576960},
  {"n": 128, "dram_bytes": 2165184, "kernel_ms": 0.37055999999999994, "flops": 34209664},
  {"n": 256, "dram_bytes": 5149312, "kernel_ms": 1.239424, "flops": 136642304},
  {"n": 512, "dram_bytes": 13409888, "kernel_ms": 4.595967999999999, "flops": 546176512},
  {"n": 1024, "dram_bytes": 44667264, "kernel_ms": 18.028736000000002, "flops": 2183920640},
  {"n": 2048, "dram_bytes": 1154626400, "kernel_ms": 71.246048, "flops": 8734111744},
  {"n": 4096, "dram_bytes": 8174034336, "kernel_ms": 283.631232, "flops": 34933305344},
  {"n": 8192, "dram_bytes": 33345201408, "kernel_ms": 1133.511072, "flops": 139726938112},
  {"n": 16384, "dram_bytes": 137583632800, "kernel_ms": 4531.887903999999, "flops": 558895185920},
  {"n": 32768, "dram_bytes": 1582854001856, "kernel_ms": 18112.942464, "flops": 2235555610624},
 ],
 "v2": [
  {"n": 64, "dram_bytes": 812960, "kernel_ms": 0.094912, "flops": 8682688},
  {"n": 128, "dram_bytes": 1755456, "kernel_ms": 0.17820799999999998, "flops": 34519424},
  {"n": 256, "dram_bytes": 4013184, "kernel_ms": 0.542656, "flops": 137456812},
  {"n": 512, "dram_bytes": 10645088, "kernel_ms": 1.9908479999999997, "flops": 548264564},
  {"n": 1024, "dram_bytes": 38345696, "kernel_ms": 7.78144, "flops": 2189099524},
  {"n": 2048, "dram_bytes": 1038482304, "kernel_ms": 30.890079999999998, "flops": 8746539776},
  {"n": 4096, "dram_bytes": 8057948480, "kernel_ms": 123.29324799999999, "flops": 34962392764},
  {"n": 8192, "dram_bytes": 32118658240, "kernel_ms": 492.70732799999996, "flops": 139793505188},
  {"n": 16384, "dram_bytes": 128266434432, "kernel_ms": 1970.4105279999999, "flops": 559044910888},
  {"n": 32768, "dram_bytes": 512102932576, "kernel_ms": 7881.08864, "flops": 2235887256464},
 ],
 "v3": [
  {"n": 64, "dram_bytes": 772768, "kernel_ms": 0.541184, "flops": 9080832},
  {"n": 128, "dram_bytes": 1528960, "kernel_ms": 1.0655999999999999, "flops": 36323328},
  {"n": 256, "dram_bytes": 3098528, "kernel_ms": 2.10688, "flops": 145293312},
  {"n": 512, "dram_bytes": 6858112, "kernel_ms": 4.186688, "flops": 581173248},
  {"n": 1024, "dram_bytes": 23032864, "kernel_ms": 16.772448, "flops": 2324692992},
  {"n": 2048, "dram_bytes": 80260288, "kernel_ms": 66.968608, "flops": 9298771968},
  {"n": 4096, "dram_bytes": 258571808, "kernel_ms": 234.156512, "flops": 37195087872},
  {"n": 8192, "dram_bytes": 917667680, "kernel_ms": 869.8391039999999, "flops": 148780351488},
  {"n": 16384, "dram_bytes": 3580210944, "kernel_ms": 3478.147424, "flops": 595121405952},
  {"n": 32768, "dram_bytes": 14153660384, "kernel_ms": 13912.305951999999, "flops": 2380485623808},
 ],
}

## Figure 1 — performance

_(placeholder: achieved TFLOP/s and GB/s vs hardware ceilings)_

In [ ]:
# Figure 1: Performance -- achieved TFLOP/s and DRAM GB/s (measured counts
# over kernel-only CUDA-event time), against the hardware ceilings.
if cuda.is_available():
    prof = DRAM_PROFILE
    pnames = {"v1": "V1 naive", "v2": "V2 tiled + online softmax",
              "v3": "V3 flash fused"}
    specs = gpu_specs()
    tmaps = {k: dict(zip(KERNEL_LENS, ktimes[pnames[k]])) for k in prof}

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for key, rows in prof.items():
        name = pnames[key]
        ns = [r["n"] for r in rows]
        tf = [r["flops"] / (tmaps[key][r["n"]] / 1e3) / 1e12 for r in rows]
        gb = [r["dram_bytes"] / (tmaps[key][r["n"]] / 1e3) / 1e9 for r in rows]
        axes[0].plot(ns, tf, marker="o", color=COLORS[name], label=name)
        axes[1].plot(ns, gb, marker="o", color=COLORS[name], label=name)
    axes[0].axhline(specs["tflops"], color="k", lw=1.5)
    axes[0].text(KERNEL_LENS[0], specs["tflops"] * 0.7,
                 f"peak fp32 {specs['tflops']:.1f} TFLOP/s", fontsize=8)
    axes[0].set_ylabel("achieved TFLOP/s")
    axes[0].set_title("compute")
    axes[1].axhline(specs["bw_gbs"], color="k", lw=1.5)
    axes[1].text(KERNEL_LENS[0], specs["bw_gbs"] * 0.7,
                 f"peak DRAM {specs['bw_gbs']:.0f} GB/s", fontsize=8)
    axes[1].set_ylabel("achieved DRAM GB/s")
    axes[1].set_title("memory bandwidth")
    for ax in axes:
        ax.set_xscale("log", base=2)
        ax.set_yscale("log")
        ax.set_xticks(KERNEL_LENS, KERNEL_LENS, rotation=45)
        ax.set_xlabel("sequence length N")
        ax.grid(True, which="both", alpha=0.3)
        ax.legend(fontsize=8)
    plt.suptitle("Figure 1: Performance (measured ops and bytes / kernel time)")
    plt.tight_layout()
    plt.show()

    print(f"{'version':>26} {'N':>6} {'GFLOP':>9} {'floor x':>8} "
          f"{'GB moved':>9} {'floor x':>8} {'TFLOP/s':>8} {'GB/s':>7}")
    for key, rows in prof.items():
        r = rows[-1]
        t_s = tmaps[key][r["n"]] / 1e3
        ff = attention_flops(cpu_model, r["n"])
        fb = attention_bytes(cpu_model, r["n"])
        print(f"{pnames[key]:>26} {r['n']:>6} {r['flops'] / 1e9:>9.1f} "
              f"{r['flops'] / ff:>7.3f}x {r['dram_bytes'] / 1e9:>9.2f} "
              f"{r['dram_bytes'] / fb:>7.1f}x {r['flops'] / t_s / 1e12:>8.2f} "
              f"{r['dram_bytes'] / t_s / 1e9:>7.1f}")
else:
    print("No CUDA device -- skipping the performance figure")

## Figure 2 — hardware utilization

_(placeholder: the same rates as % of each ceiling)_

In [ ]:
# Figure 2: Hardware utilization -- the same measured rates as % of each
# ceiling (fp32 compute peak, DRAM bandwidth peak).
if cuda.is_available():
    prof = DRAM_PROFILE
    pnames = {"v1": "V1 naive", "v2": "V2 tiled + online softmax",
              "v3": "V3 flash fused"}
    specs = gpu_specs()
    tmaps = {k: dict(zip(KERNEL_LENS, ktimes[pnames[k]])) for k in prof}

    fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
    for key, rows in prof.items():
        name = pnames[key]
        ns = [r["n"] for r in rows]
        cu = [100 * r["flops"] / (tmaps[key][r["n"]] / 1e3) / 1e12 / specs["tflops"]
              for r in rows]
        bu = [100 * r["dram_bytes"] / (tmaps[key][r["n"]] / 1e3) / 1e9 / specs["bw_gbs"]
              for r in rows]
        axes[0].plot(ns, cu, marker="o", color=COLORS[name], label=name)
        axes[1].plot(ns, bu, marker="o", color=COLORS[name], label=name)
    axes[0].set_title("compute utilization")
    axes[1].set_title("memory bandwidth utilization")
    axes[0].set_ylabel("% of peak")
    for ax in axes:
        ax.axhline(100.0, color="k", lw=1.5)
        ax.set_xscale("log", base=2)
        ax.set_yscale("log")
        ax.set_xticks(KERNEL_LENS, KERNEL_LENS, rotation=45)
        ax.set_xlabel("sequence length N")
        ax.grid(True, which="both", alpha=0.3)
        ax.legend(fontsize=8)
    plt.suptitle("Figure 2: Hardware utilization (100% = ceiling)")
    plt.tight_layout()
    plt.show()

    print(f"{'version':>26} {'N':>6} {'compute %':>10} {'bandwidth %':>12}")
    for key, rows in prof.items():
        r = rows[-1]
        t_s = tmaps[key][r["n"]] / 1e3
        cu = 100 * r["flops"] / t_s / 1e12 / specs["tflops"]
        bu = 100 * r["dram_bytes"] / t_s / 1e9 / specs["bw_gbs"]
        print(f"{pnames[key]:>26} {r['n']:>6} {cu:>9.1f}% {bu:>11.1f}%")
else:
    print("No CUDA device -- skipping the utilization figure")

## Figure 3 — arithmetic intensity

_(placeholder: measured FLOP/byte vs algorithmic AI and the ridge)_

In [ ]:
# Figure 3: Arithmetic intensity -- measured FLOPs per measured DRAM byte,
# vs the algorithmic AI (floor FLOPs / compulsory bytes = N/4) and the ridge.
if cuda.is_available():
    prof = DRAM_PROFILE
    pnames = {"v1": "V1 naive", "v2": "V2 tiled + online softmax",
              "v3": "V3 flash fused"}
    specs = gpu_specs()
    fig, ax = plt.subplots(figsize=(8, 5))
    for key, rows in prof.items():
        name = pnames[key]
        ax.plot([r["n"] for r in rows],
                [r["flops"] / r["dram_bytes"] for r in rows],
                marker="o", color=COLORS[name], label=name)
    alg = [attention_flops(cpu_model, n) / attention_bytes(cpu_model, n)
           for n in KERNEL_LENS]
    ax.plot(KERNEL_LENS, alg, "k:", label="algorithmic AI (floor FLOPs / compulsory bytes)")
    ridge = specs["tflops"] * 1e12 / (specs["bw_gbs"] * 1e9)
    ax.axhline(ridge, color="k", lw=1.5)
    ax.text(KERNEL_LENS[0], ridge * 1.25,
            f"ridge {ridge:.0f} FLOP/byte: compute-bound above, memory-bound below",
            fontsize=8)
    ax.set_xscale("log", base=2)
    ax.set_yscale("log")
    ax.set_xticks(KERNEL_LENS, KERNEL_LENS, rotation=45)
    ax.set_xlabel("sequence length N")
    ax.set_ylabel("arithmetic intensity (FLOP / DRAM byte)")
    ax.set_title("Figure 3: Arithmetic intensity (measured)")
    ax.grid(True, which="both", alpha=0.3)
    ax.legend(fontsize=8)
    plt.tight_layout()
    plt.show()
else:
    print("No CUDA device -- skipping the arithmetic-intensity figure")

## Figure 4 — memory footprint

_(placeholder: allocator high-water mark per pass, swept to OOM)_

In [ ]:
# Figure 4: peak extra GPU memory of one attention pass, swept until the
# O(N^2) versions hit the card's capacity. None (OOM) ends a line.
if cuda.is_available():
    FOOT_LENS = [2 ** x for x in range(6, 17)]  # 64 .. 65536
    foot_models = {"V1 naive": GpuV1(), "V2 tiled + online softmax": GpuV2(),
                   "V3 flash fused": GpuV3()}
    foot = {name: [] for name in foot_models}
    for n in FOOT_LENS:
        for name, m in foot_models.items():
            foot[name].append(attention_peak_extra_bytes(m, n))
        torch.cuda.empty_cache()

    fig, ax = plt.subplots(figsize=(8, 5))
    for name, vals in foot.items():
        pts = [(n, b) for n, b in zip(FOOT_LENS, vals) if b is not None]
        xs, ys = [p[0] for p in pts], [p[1] / 1e6 for p in pts]
        ax.plot(xs, ys, marker="o", color=COLORS[name], label=name)
        if len(pts) < len(FOOT_LENS):  # line ends where the alloc stopped fitting
            ax.text(xs[-1] * 1.2, ys[-1], f"OOM @ {FOOT_LENS[len(pts)]}",
                    color=COLORS[name], fontsize=8, va="center")
    total = torch.cuda.get_device_properties(0).total_memory
    ax.axhline(total / 1e6, color="k", lw=1.5)
    ax.text(FOOT_LENS[0], total / 1e6 * 1.3, f"device memory {total / 1e9:.0f} GB",
            fontsize=8)
    ax.set_xscale("log", base=2)
    ax.set_yscale("log")
    ax.set_xticks(FOOT_LENS, FOOT_LENS, rotation=45)
    ax.set_xlabel("sequence length N")
    ax.set_ylabel("peak extra memory per pass (MB)")
    ax.set_title("Figure 4: Attention memory footprint (measured allocator high-water mark)")
    ax.grid(True, which="both", alpha=0.3)
    ax.legend()
    plt.tight_layout()
    plt.show()

    for name, vals in foot.items():
        fit = sum(b is not None for b in vals)
        print(f"{name:>26}: fits to N = {FOOT_LENS[fit - 1]}"
              + ("" if fit == len(FOOT_LENS) else f", OOM at N = {FOOT_LENS[fit]}"))
else:
    print("No CUDA device -- skipping the footprint sweep")